In [1]:
%gui qt
import napari
import numpy as np
import tifffile
import scipy as sp
import cv2
import scipy.ndimage as ndimage

In [2]:
viewer = napari.Viewer()

In [3]:
data = 'S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20220107_MER_IMARE-110171_19_55_48hpa_C4/cjw/gcs_etomo/DeepLearn/'

In [4]:
img = tifffile.imread(data + 'Output.tif')
#img = np.reshape(img, [int(img.shape[0]/3),3,img.shape[1],img.shape[2]])

In [5]:
img.shape

(1701, 3, 2212, 4508)

In [6]:
#img[:,1,:,:] = img[:,2,:,:] - img[:,1,:,:]

In [7]:
test = np.swapaxes(img,0,2)

In [8]:
rslt = np.array([cv2.GaussianBlur(a[2,:,:],(0,0), sigmaY=2.0, sigmaX=0.001) for a in test])

In [9]:
rslt = np.swapaxes(rslt,0,1)

In [10]:
rslt = rslt>128

In [11]:
viewer.add_image(img[:,2,:,:])

<Image layer 'Image' at 0x15b67b8fc40>

In [12]:
viewer.add_image(rslt)

<Image layer 'rslt' at 0x15b67b8f610>

In [13]:
viewer.add_image(img[:,0,:,:])

<Image layer 'Image [1]' at 0x15b726ce100>

In [14]:
labeled_array, num_features = ndimage.label(rslt)

In [15]:
del img
del test
del rslt

In [16]:
counts = ndimage.labeled_comprehension(labeled_array, labeled_array, np.arange(1,num_features+1), len, float, 0)

In [17]:
np.save('Labels.npy', labeled_array)
np.save('Counts.npy', counts)

In [18]:
lst = np.arange(0,num_features+1)

In [19]:
big_enough = lst
big_enough[1:] = (counts>50000) * lst[1:]

In [20]:
size_filtered = big_enough[labeled_array]

In [21]:
size_filtered.shape

(1701, 2212, 4508)

In [22]:
viewer.add_image(size_filtered, colormap='magenta', blending='additive')

<Image layer 'size_filtered' at 0x15b73413be0>

In [23]:
size_filtered_mask = size_filtered>5

In [24]:
viewer.add_image(size_filtered_mask, colormap='magenta', blending='additive')

<Image layer 'size_filtered_mask' at 0x15b73f25250>

In [25]:
viewer.add_image(img[:,0,:,:], colormap='gray', blending='additive')

NameError: name 'img' is not defined

In [26]:
np.save('Size_Filtered.npy', size_filtered_mask)

In [ ]:
del labeled_array
del size_filtered


In [27]:
real_labels, nfs = ndimage.label(size_filtered_mask)

In [28]:
np.save(data+'Filtered_Labels.npy', real_labels)

In [29]:
tifffile.imsave(data+'Labels.tif', real_labels.astype(np.ushort), imagej=True)

C:\Users\smc\AppData\Local\Temp\ipykernel_4404\1743038886.py:1: DeprecationWarning: <tifffile.imsave> is deprecated. Use tifffile.imwrite
  tifffile.imsave(data+'Labels.tif', real_labels.astype(np.ushort), imagej=True)
C:\Users\smc\Miniconda3\envs\ip\lib\site-packages\tifffile\tifffile.py:2972: UserWarning: <tifffile.TiffWriter 'Labels.tif'> truncating ImageJ file
  warnings.warn(f'{self!r} truncating ImageJ file', UserWarning)


In [2]:
img = tifffile.imread('S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Output.tif')
labels = tifffile.imread('S:/micro/asa/stn/mel/20-15_Irradiation_Datasets/20210721_MER_IMARE-106746_20_15_1A_control/20-15-1A_after_crash/cjw/DeepLearn/Labels.tif')

In [5]:
ds = rescale(img[:,0,:,:], [0.5, 0.5, 0.5])

In [6]:
dslab = rescale(labels[:,:,:], [0.5, 0.5, 0.5])

In [7]:
viewer.add_image(ds, blending='additive', colormap='gray')


TypeError: Only integer types are supported for Labels layers, but data contains float64.

In [8]:
viewer.add_image(dslab, blending='additive', colormap='magenta')

<Image layer 'dslab' at 0x22b3e4313c8>

In [3]:
real_labels = np.load('Filtered_Labels.npy')

In [4]:
viewer.add_labels(real_labels, scale=[60, 60, 60])

<Labels layer 'real_labels' at 0x24d45f0f3c8>